# DDIM images to random pixel affines to invert to sign-template latent affine

Same end-to-end SD1.5 pipeline as `ddim_affine_sign_registration_n5.ipynb`, except the
non-identity warps are **random 6-parameter affines drawn uniformly inside the
registration search box** (not hand-picked axis-aligned shifts / mild compounds).

1. Draw 5 Gaussian latents `z`, generate images.
2. Keep an identity **baseline**, then apply `NUM_RANDOM_AFFINES` random pixel warps.
3. DDIM-invert each warped image to `z_hat`.
4. Estimate the affine in latent space with the **sign-only baseline template** vs full
   signed observation, plus a **random 0/1 control**.
5. Score bit accuracy, correlations, and parameter error vs the ideal latent map
   (`t_px / 8`; linear params unchanged).
6. Save images, latents, tables, and figures under
   `results/ddim_affine_sign_registration_random_n5/`.

### Random draw

Latent parameters are sampled in:

- angle ∈ [-20°, 20°]
- scale_x, scale_y ∈ [0.85, 1.15]
- shear ∈ [-0.2, 0.2]
- tx, ty (latent) ∈ [-4, 4]

Pixel translations used for warping are `t_latent * VAE_SCALE_FACTOR` so every draw
stays inside the L-BFGS search bounds after the ideal map.

### HiPerGator (OOD Jupyter)

- Kernel: `gaussian-rotation`
- **One GPU**, ~8 GB, account/QoS `eel5840`, partition `hpg-turin`
- Working directory: **repo root**
- Budget: 5 samples × (1 baseline + 4 random) inversions. Target **under 10 minutes**.
- Re-running skips samples whose `.pt` files are already complete.


## 1. Configuration


In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import ndimage
from scipy.optimize import minimize
from scipy.stats import pearsonr

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

NUM_SAMPLES = 5
NUM_RANDOM_AFFINES = 4
SEED = 42
REFERENCE_SAMPLE_COUNT = 100
OUTPUT_DIR = Path("results/ddim_affine_sign_registration_random_n5")
LATENT_DIR = OUTPUT_DIR / "latents"
IMAGE_DIR = OUTPUT_DIR / "images"
FIGURES_DIR = OUTPUT_DIR / "figures"

MODEL_ID = "runwayml/stable-diffusion-v1-5"
NUM_INFERENCE_STEPS = 50
DEVICE = "cuda"
DTYPE = "float16"
IMAGE_SIZE = 512
PROMPT = "a photograph of an astronaut riding a horse"
GUIDANCE_SCALE = 7.5
VAE_SCALE_FACTOR = 8

PARAM_NAMES = ["angle_deg", "scale_x", "scale_y", "shear", "tx", "ty"]
ANGLE_RANGE = (-20.0, 20.0)
SCALE_RANGE = (0.85, 1.15)
SHEAR_RANGE = (-0.2, 0.2)
TRANSLATION_RANGE = (-4.0, 4.0)  # latent units; pixel tx/ty = latent * 8
BOUNDS = [ANGLE_RANGE, SCALE_RANGE, SCALE_RANGE, SHEAR_RANGE, TRANSLATION_RANGE, TRANSLATION_RANGE]
LB = np.array([b[0] for b in BOUNDS], dtype=float)
UB = np.array([b[1] for b in BOUNDS], dtype=float)
SCIPY_BOUNDS = list(zip(LB, UB))

WARP_ORDER = 1
VALID_RADIUS_FRAC = 0.32
COARSE_ANGLE_STEP = 4.0
COARSE_TRANSLATION_STEP = 1.0


def sample_latent_affine(rng: np.random.Generator) -> np.ndarray:
    return np.array(
        [
            rng.uniform(*ANGLE_RANGE),
            rng.uniform(*SCALE_RANGE),
            rng.uniform(*SCALE_RANGE),
            rng.uniform(*SHEAR_RANGE),
            rng.uniform(*TRANSLATION_RANGE),
            rng.uniform(*TRANSLATION_RANGE),
        ],
        dtype=float,
    )


def latent_to_pixel_params(latent_params) -> tuple[float, float, float, float, float, float]:
    angle, sx, sy, shear, tx, ty = np.asarray(latent_params, dtype=float)
    return (
        float(angle),
        float(sx),
        float(sy),
        float(shear),
        float(tx * VAE_SCALE_FACTOR),
        float(ty * VAE_SCALE_FACTOR),
    )


def build_conditions(n_random: int = NUM_RANDOM_AFFINES) -> list[dict]:
    # Shared across samples so summary_by_condition stays meaningful.
    rng = np.random.default_rng(SEED + 123)
    conditions = [{"name": "baseline", "params": (0.0, 1.0, 1.0, 0.0, 0.0, 0.0)}]
    for index in range(n_random):
        latent = sample_latent_affine(rng)
        conditions.append(
            {
                "name": f"rand_{index:02d}",
                "params": latent_to_pixel_params(latent),
            }
        )
    return conditions


CONDITIONS = build_conditions()

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})
for path in (OUTPUT_DIR, LATENT_DIR, IMAGE_DIR, FIGURES_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"samples={NUM_SAMPLES}, conditions={len(CONDITIONS)} (baseline + {NUM_RANDOM_AFFINES} random)")
print(f"inversions if nothing cached: {NUM_SAMPLES * len(CONDITIONS)}")
print(f"output -> {OUTPUT_DIR.resolve()}")
print("drawn conditions (pixel params = angle, sx, sy, shear, tx_px, ty_px):")
for cond in CONDITIONS:
    print(f"  {cond['name']}: {tuple(round(v, 4) for v in cond['params'])}")


## 2. Geometry helpers

Pixel warps use the same six-parameter map as `affine_registration.ipynb`. Expected **latent** parameters keep the linear part and divide translations by the VAE scale (8). Registration maximizes multi-channel correlation of the warped sign template with the full signed observation.


In [ ]:
def fmt_duration(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    if seconds < 60:
        return f"{seconds:.0f}s"
    minutes, secs = divmod(int(round(seconds)), 60)
    if minutes < 60:
        return f"{minutes}m{secs:02d}s"
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}m"


class StepClock:
    def __init__(self, total: int, label: str):
        self.total = total
        self.label = label
        self.done = 0
        self.started = time.perf_counter()
        self.durations: list[float] = []
        self._tick = self.started

    def begin(self) -> None:
        self._tick = time.perf_counter()

    def end(self, detail: str) -> None:
        now = time.perf_counter()
        duration = now - self._tick
        self.durations.append(duration)
        self.done += 1
        average = sum(self.durations) / len(self.durations)
        remaining = self.total - self.done
        eta = average * remaining
        print(
            f"[{self.label} {self.done}/{self.total}] {detail} | "
            f"this {duration:.1f}s avg {average:.1f}s | "
            f"elapsed {fmt_duration(now - self.started)} | "
            f"~{fmt_duration(eta)} left",
            flush=True,
        )


def sample_seeds() -> list[int]:
    rng = np.random.default_rng(SEED)
    return [int(s) for s in rng.integers(0, 2**31 - 1, size=REFERENCE_SAMPLE_COUNT, endpoint=False)[:NUM_SAMPLES]]


def latent_key(name: str) -> str:
    return f"z_hat__{name}"


def sample_path(sample_id: int) -> Path:
    return LATENT_DIR / f"sample_{sample_id:03d}.pt"


def sample_image_dir(sample_id: int) -> Path:
    path = IMAGE_DIR / f"sample_{sample_id:03d}"
    path.mkdir(parents=True, exist_ok=True)
    return path


def sample_is_complete(sample_id: int) -> bool:
    path = sample_path(sample_id)
    if not path.is_file():
        return False
    blob = torch.load(path, map_location="cpu", weights_only=True)
    needed = {"z", "image_seed"} | {latent_key(c["name"]) for c in CONDITIONS}
    return needed.issubset(blob.keys())


def params_to_matrix(params) -> tuple[np.ndarray, np.ndarray]:
    angle, sx, sy, shear, tx, ty = np.asarray(params, dtype=float)
    theta = np.deg2rad(angle)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    rotation = np.array([[cos_t, -sin_t], [sin_t, cos_t]], dtype=float)
    linear = np.array([[sx, sx * shear], [0.0, sy]], dtype=float)
    return rotation @ linear, np.array([tx, ty], dtype=float)


def image_center(spatial_shape: tuple[int, int]) -> np.ndarray:
    return np.array(spatial_shape, dtype=float) / 2.0 - 0.5


def warp_array(image: np.ndarray, matrix: np.ndarray, translation: np.ndarray) -> np.ndarray:
    # Warp HWC or CHW float array with a shared 2-D affine (inverse map).
    if image.ndim == 3 and image.shape[0] in (1, 3, 4) and image.shape[-1] not in (1, 3):
        channels = image
        spatial = image.shape[-2:]
        stack_axis = 0
    elif image.ndim == 3:
        channels = np.moveaxis(image, -1, 0)
        spatial = image.shape[:2]
        stack_axis = -1
    elif image.ndim == 2:
        channels = image[None]
        spatial = image.shape
        stack_axis = 0
    else:
        raise ValueError(f"unsupported image shape {image.shape}")

    center = image_center(spatial)
    inverse = np.linalg.inv(matrix)
    affine = np.zeros((2, 3), dtype=float)
    affine[:, :2] = inverse
    affine[:, 2] = -inverse @ (center + translation) + center
    warped = np.stack(
        [
            ndimage.affine_transform(
                channel,
                affine,
                order=WARP_ORDER,
                mode="constant",
                cval=0.0,
                prefilter=True,
            )
            for channel in channels
        ],
        axis=0,
    )
    if stack_axis == -1:
        return np.moveaxis(warped, 0, -1)
    if image.ndim == 2:
        return warped[0]
    return warped


def warp_pil(image: Image.Image, params) -> Image.Image:
    rgb = np.asarray(image.convert("RGB"), dtype=np.float64) / 255.0
    warped = np.clip(warp_array(rgb, *params_to_matrix(params)), 0.0, 1.0)
    return Image.fromarray((warped * 255.0).round().astype(np.uint8), mode="RGB")


def expected_latent_params(pixel_params) -> np.ndarray:
    angle, sx, sy, shear, tx, ty = np.asarray(pixel_params, dtype=float)
    return np.array(
        [angle, sx, sy, shear, tx / VAE_SCALE_FACTOR, ty / VAE_SCALE_FACTOR],
        dtype=float,
    )


def as_chw(tensor) -> np.ndarray:
    array = np.asarray(tensor, dtype=np.float64)
    if array.ndim == 4 and array.shape[0] == 1:
        array = array[0]
    if array.ndim != 3:
        raise ValueError(f"expected (C,H,W), got {array.shape}")
    return array


def invert_params(params: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    matrix, translation = params_to_matrix(params)
    inverse = np.linalg.inv(matrix)
    return inverse, -inverse @ translation


def circular_valid_mask(spatial_shape: tuple[int, int], radius_frac: float = VALID_RADIUS_FRAC) -> np.ndarray:
    height, width = spatial_shape
    center_y, center_x = image_center(spatial_shape)
    rows, cols = np.mgrid[0:height, 0:width]
    radius = radius_frac * float(min(height, width))
    return (rows - center_y) ** 2 + (cols - center_x) ** 2 <= radius ** 2


def bits_from_latent(latent: np.ndarray) -> np.ndarray:
    return (np.asarray(latent) >= 0).astype(np.float64)


def bits_to_pm(bits01: np.ndarray) -> np.ndarray:
    return 2.0 * bits01 - 1.0


def sign_pm(latent: np.ndarray) -> np.ndarray:
    return np.where(np.asarray(latent) >= 0, 1.0, -1.0)


def masked_vectors(a: np.ndarray, b: np.ndarray, mask: np.ndarray):
    return a[:, mask].ravel(), b[:, mask].ravel()


def full_latent_corr(a: np.ndarray, b: np.ndarray, mask: np.ndarray | None = None) -> float:
    if mask is None:
        left, right = a.ravel(), b.ravel()
    else:
        left, right = masked_vectors(a, b, mask)
    if left.size < 2 or np.std(left) < 1e-12 or np.std(right) < 1e-12:
        return 0.0
    return float(pearsonr(left, right).statistic)


def bit_metrics(bits01: np.ndarray, observation: np.ndarray, params: np.ndarray, mask: np.ndarray) -> dict:
    bits_pm = bits_to_pm(bits01)
    forward = warp_array(bits_pm, *params_to_matrix(params))
    back = warp_array(observation, *invert_params(params))
    signed_back = sign_pm(back)
    bits_m, signed_m = masked_vectors(bits_pm, signed_back, mask)
    _, back_m = masked_vectors(bits_pm, back, mask)
    return {
        "forward_correlation": full_latent_corr(forward, observation, mask),
        "bit_accuracy": float(np.mean(bits_m == signed_m)),
        "bit_mse": float(np.mean((bits_m - signed_m) ** 2)),
        "bit_correlation": full_latent_corr(bits_pm, signed_back, mask),
        "continuous_correlation": full_latent_corr(bits_pm, back, mask),
        "continuous_mse": float(np.mean((bits_m - back_m) ** 2)),
    }


def condition_title(cond: dict) -> str:
    if cond["name"] == "baseline":
        return "baseline"
    angle, sx, sy, shear, tx, ty = cond["params"]
    return (
        f"{cond['name']}\n"
        f"θ={angle:.1f}° sx={sx:.2f} sy={sy:.2f}\n"
        f"k={shear:.2f} t=({tx:.1f},{ty:.1f})px"
    )


print("conditions (pixel -> expected latent):")
for cond in CONDITIONS:
    exp = expected_latent_params(cond["params"])
    print(
        f"  {cond['name']}: pixel={tuple(round(float(v), 4) for v in cond['params'])} "
        f"-> latent=({exp[0]:.2f}°, {exp[1]:.3f}, {exp[2]:.3f}, {exp[3]:.3f}, "
        f"{exp[4]:.3f}, {exp[5]:.3f})"
    )


In [ ]:
def forward_correlation(bits_pm: np.ndarray, observation: np.ndarray, params: np.ndarray, mask: np.ndarray) -> float:
    warped = warp_array(bits_pm, *params_to_matrix(params))
    return full_latent_corr(warped, observation, mask)


def coarse_initialize(bits_pm: np.ndarray, observation: np.ndarray, mask: np.ndarray) -> np.ndarray:
    identity = np.array([0.0, 1.0, 1.0, 0.0, 0.0, 0.0], dtype=float)
    best_params = identity.copy()
    best_corr = forward_correlation(bits_pm, observation, best_params, mask)

    for angle in np.arange(ANGLE_RANGE[0], ANGLE_RANGE[1] + 0.5 * COARSE_ANGLE_STEP, COARSE_ANGLE_STEP):
        params = np.array([angle, 1.0, 1.0, 0.0, 0.0, 0.0], dtype=float)
        corr = forward_correlation(bits_pm, observation, params, mask)
        if corr > best_corr:
            best_corr, best_params = corr, params

    for tx in np.arange(TRANSLATION_RANGE[0], TRANSLATION_RANGE[1] + 0.5 * COARSE_TRANSLATION_STEP, COARSE_TRANSLATION_STEP):
        for ty in np.arange(TRANSLATION_RANGE[0], TRANSLATION_RANGE[1] + 0.5 * COARSE_TRANSLATION_STEP, COARSE_TRANSLATION_STEP):
            params = np.array([0.0, 1.0, 1.0, 0.0, tx, ty], dtype=float)
            corr = forward_correlation(bits_pm, observation, params, mask)
            if corr > best_corr:
                best_corr, best_params = corr, params

    angle0, _, _, _, tx0, ty0 = best_params
    for angle in np.unique(np.clip([angle0 - COARSE_ANGLE_STEP, angle0, angle0 + COARSE_ANGLE_STEP], *ANGLE_RANGE)):
        for tx in np.unique(np.clip([tx0 - COARSE_TRANSLATION_STEP, tx0, tx0 + COARSE_TRANSLATION_STEP], *TRANSLATION_RANGE)):
            for ty in np.unique(np.clip([ty0 - COARSE_TRANSLATION_STEP, ty0, ty0 + COARSE_TRANSLATION_STEP], *TRANSLATION_RANGE)):
                params = np.array([angle, 1.0, 1.0, 0.0, tx, ty], dtype=float)
                corr = forward_correlation(bits_pm, observation, params, mask)
                if corr > best_corr:
                    best_corr, best_params = corr, params
    return best_params


def register_sign_affine(bits01: np.ndarray, observation: np.ndarray, mask: np.ndarray) -> dict:
    started = time.perf_counter()
    bits_pm = bits_to_pm(bits01)
    init = coarse_initialize(bits_pm, observation, mask)

    def objective(params: np.ndarray) -> float:
        return -forward_correlation(bits_pm, observation, params, mask)

    refined = minimize(
        objective,
        init,
        method="L-BFGS-B",
        bounds=SCIPY_BOUNDS,
        options={"ftol": 1e-9, "gtol": 1e-7, "maxiter": 200},
    )
    estimate = np.asarray(refined.x, dtype=float)
    metrics = bit_metrics(bits01, observation, estimate, mask)
    return {
        "estimated_params": estimate,
        "method": "coarse+L-BFGS-B",
        "runtime_s": time.perf_counter() - started,
        **metrics,
    }


print("registration helpers ready")


## 3. Generate images, warp, invert, and save

Loads generation, produces all pending images, releases VRAM, then loads inversion. Every sample writes `latents/sample_XXX.pt` and PNGs under `images/sample_XXX/`.


In [ ]:
def run_generate_and_invert() -> None:
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. Use a one-GPU HiPerGator Jupyter session with the gaussian-rotation kernel."
        )
    from experiments.generate_and_invert import (
        _dtype,
        _git_commit,
        _load_generation_pipeline,
        _load_inversion_pipeline,
        _recovered_latent,
        _release_pipeline,
        _sample_latent,
    )

    seeds = sample_seeds()
    pending = [i for i in range(NUM_SAMPLES) if not sample_is_complete(i)]
    print(
        f"{NUM_SAMPLES - len(pending)}/{NUM_SAMPLES} samples already complete; "
        f"{len(pending)} still need generation+inversion.",
        flush=True,
    )
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
    if not pending:
        print("Nothing to invert.", flush=True)
        return

    dtype = _dtype(DTYPE)
    load_started = time.perf_counter()
    print("Loading generation pipeline...", flush=True)
    pipe = _load_generation_pipeline(MODEL_ID, dtype, DEVICE)
    print(f"Generation ready in {fmt_duration(time.perf_counter() - load_started)}.", flush=True)
    if float(pipe.scheduler.init_noise_sigma) != 1.0:
        raise RuntimeError("DDIM init_noise_sigma is not 1.")
    if pipe.vae_scale_factor != VAE_SCALE_FACTOR:
        raise RuntimeError(f"VAE scale {pipe.vae_scale_factor} != {VAE_SCALE_FACTOR}")

    generated = []
    gen_clock = StepClock(len(pending), "generation")
    try:
        for sample_id in pending:
            gen_clock.begin()
            z = _sample_latent(pipe, seeds[sample_id], IMAGE_SIZE, IMAGE_SIZE, dtype)
            result = pipe(
                prompt=PROMPT,
                latents=z.to(DEVICE),
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                height=IMAGE_SIZE,
                width=IMAGE_SIZE,
            )
            if len(result.images) != 1:
                raise RuntimeError(f"sample {sample_id} returned {len(result.images)} images")
            image = result.images[0].convert("RGB")
            if image.size != (IMAGE_SIZE, IMAGE_SIZE):
                raise RuntimeError(f"bad image size {image.size}")
            generated.append((sample_id, z.detach().to(device="cpu", dtype=torch.float32), image))
            gen_clock.end(f"sample {sample_id}, seed {seeds[sample_id]}")
    finally:
        _release_pipeline(pipe)
        print("Generation pipeline released.", flush=True)

    load_started = time.perf_counter()
    print("Loading inversion pipeline...", flush=True)
    inverse = _load_inversion_pipeline(MODEL_ID, dtype, DEVICE)
    print(f"Inversion ready in {fmt_duration(time.perf_counter() - load_started)}.", flush=True)
    if type(inverse.inverse_scheduler).__name__ != "DDIMInverseScheduler":
        raise RuntimeError(f"unexpected inverse scheduler {type(inverse.inverse_scheduler).__name__}")

    inv_clock = StepClock(len(generated) * len(CONDITIONS), "inversion")
    try:
        for sample_id, z, image in generated:
            blob = {"z": z, "image_seed": torch.tensor(seeds[sample_id], dtype=torch.int64)}
            out_img = sample_image_dir(sample_id)
            image.save(out_img / "original.png")
            sample_started = time.perf_counter()
            for cond_index, cond in enumerate(CONDITIONS):
                inv_clock.begin()
                name = cond["name"]
                params = np.asarray(cond["params"], dtype=float)
                torch.manual_seed(SEED + 1000 * sample_id + cond_index)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(SEED + 1000 * sample_id + cond_index)
                warped = warp_pil(image, params) if name != "baseline" else image.copy()
                warped.save(out_img / f"{name}.png")
                inverted = inverse.invert(
                    prompt=PROMPT,
                    image=warped,
                    num_inference_steps=NUM_INFERENCE_STEPS,
                    guidance_scale=GUIDANCE_SCALE,
                    inpaint_strength=1.0,
                    num_reg_steps=0,
                )
                z_hat = _recovered_latent(inverted.latents, z.shape)
                z_hat = z_hat.detach().to(device="cpu", dtype=torch.float32).contiguous()
                if tuple(z_hat.shape) != tuple(z.shape) or not torch.isfinite(z_hat).all():
                    raise RuntimeError(f"bad inverted latent sample {sample_id} {name}")
                blob[latent_key(name)] = z_hat
                inv_clock.end(f"sample {sample_id} {name}")
            torch.save(blob, sample_path(sample_id))
            print(
                f"Saved {sample_path(sample_id)} "
                f"({fmt_duration(time.perf_counter() - sample_started)} for {len(CONDITIONS)} inversions).",
                flush=True,
            )
    finally:
        _release_pipeline(inverse)
        print("Inversion pipeline released.", flush=True)

    meta = {
        "seed": SEED,
        "num_samples": NUM_SAMPLES,
        "num_random_affines": NUM_RANDOM_AFFINES,
        "sample_seeds": sample_seeds(),
        "samples_complete": [i for i in range(NUM_SAMPLES) if sample_is_complete(i)],
        "model_id": MODEL_ID,
        "image_size": IMAGE_SIZE,
        "vae_scale_factor": VAE_SCALE_FACTOR,
        "num_inference_steps": NUM_INFERENCE_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "prompt": PROMPT,
        "dtype": DTYPE,
        "search_bounds_latent": {
            "angle_deg": list(ANGLE_RANGE),
            "scale": list(SCALE_RANGE),
            "shear": list(SHEAR_RANGE),
            "translation": list(TRANSLATION_RANGE),
        },
        "conditions": [
            {
                "name": c["name"],
                "params_pixel": list(map(float, c["params"])),
                "params_latent_expected": list(map(float, expected_latent_params(c["params"]))),
            }
            for c in CONDITIONS
        ],
        "git_commit": _git_commit(),
        "note": "random pixel affines in search box, then DDIM invert; sign-template registration in later cells",
    }
    (OUTPUT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")
    print(f"wrote {OUTPUT_DIR / 'metadata.json'}", flush=True)


run_started = time.perf_counter()
run_generate_and_invert()
print(f"generate+invert wall {fmt_duration(time.perf_counter() - run_started)}", flush=True)


## 4. Example warped images (sample 0)


In [ ]:
example_dir = sample_image_dir(0)
missing = [c["name"] for c in CONDITIONS if not (example_dir / f"{c['name']}.png").is_file()]
if missing:
    raise FileNotFoundError(f"missing sample 0 images {missing}; run the generate/invert cell first")

fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(3.4 * len(CONDITIONS), 4.0))
for ax, cond in zip(axes, CONDITIONS):
    img = Image.open(example_dir / f"{cond['name']}.png")
    ax.imshow(img)
    ax.set_title(condition_title(cond), fontsize=8)
    ax.axis("off")
fig.suptitle("Sample 0: generation after each random pixel affine (baseline = unwarped)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "example_warped_images_sample0.png", bbox_inches="tight")
plt.show()


## 5. Sign-template latent affine registration (+ random control)


In [ ]:
incomplete = [i for i in range(NUM_SAMPLES) if not sample_is_complete(i)]
if incomplete:
    raise FileNotFoundError(f"incomplete samples {incomplete}; finish generate/invert first")

blobs = [torch.load(sample_path(i), map_location="cpu", weights_only=True) for i in range(NUM_SAMPLES)]
SHAPE = as_chw(blobs[0]["z"]).shape
VALID_MASK = circular_valid_mask(SHAPE[-2:])
rng = np.random.default_rng(SEED + 7)
random_bits_by_sample = {i: (rng.random(SHAPE) < 0.5).astype(np.float64) for i in range(NUM_SAMPLES)}

rows = []
reg_started = time.perf_counter()
jobs = [(sid, cond) for sid in range(NUM_SAMPLES) for cond in CONDITIONS]
print(f"registering {len(jobs)} (sample, condition) pairs x matched+random...", flush=True)

for index, (sample_id, cond) in enumerate(jobs, start=1):
    blob = blobs[sample_id]
    z = as_chw(blob["z"])
    baseline = as_chw(blob[latent_key("baseline")])
    observation = as_chw(blob[latent_key(cond["name"])])
    expected = expected_latent_params(cond["params"])
    matched_bits = bits_from_latent(baseline)
    random_bits = random_bits_by_sample[sample_id]

    if cond["name"] == "baseline":
        matched = {
            "estimated_params": expected.copy(),
            "method": "identity",
            "runtime_s": 0.0,
            **bit_metrics(matched_bits, observation, expected, VALID_MASK),
        }
        random = {
            "estimated_params": expected.copy(),
            "method": "identity",
            "runtime_s": 0.0,
            **bit_metrics(random_bits, observation, expected, VALID_MASK),
        }
    else:
        matched = register_sign_affine(matched_bits, observation, VALID_MASK)
        random = register_sign_affine(random_bits, observation, VALID_MASK)

    random_at_true = bit_metrics(random_bits, observation, expected, VALID_MASK)
    identity = np.array([0.0, 1.0, 1.0, 0.0, 0.0, 0.0], dtype=float)
    true_distance = float(np.linalg.norm((expected - identity) / np.maximum(UB - LB, 1e-8)))
    row = {
        "sample_id": sample_id,
        "condition": cond["name"],
        "matched_method": matched["method"],
        "random_method": random["method"],
        "matched_runtime_s": matched["runtime_s"],
        "random_runtime_s": random["runtime_s"],
        "matched_bit_accuracy": matched["bit_accuracy"],
        "matched_bit_correlation": matched["bit_correlation"],
        "matched_continuous_correlation": matched["continuous_correlation"],
        "matched_forward_correlation": matched["forward_correlation"],
        "random_bit_accuracy": random["bit_accuracy"],
        "random_bit_correlation": random["bit_correlation"],
        "random_forward_correlation": random["forward_correlation"],
        "random_bit_accuracy_at_true": random_at_true["bit_accuracy"],
        "matched_param_mae": float(np.mean(np.abs(matched["estimated_params"] - expected))),
        "random_param_mae": float(np.mean(np.abs(random["estimated_params"] - expected))),
        "matched_angle_err": float(abs(matched["estimated_params"][0] - expected[0])),
        "matched_tx_err": float(abs(matched["estimated_params"][4] - expected[4])),
        "matched_ty_err": float(abs(matched["estimated_params"][5] - expected[5])),
        "random_angle_err": float(abs(random["estimated_params"][0] - expected[0])),
        "random_tx_err": float(abs(random["estimated_params"][4] - expected[4])),
        "random_ty_err": float(abs(random["estimated_params"][5] - expected[5])),
        "true_param_distance": true_distance,
        "corr_zhat_vs_z": full_latent_corr(observation, z, VALID_MASK),
    }
    for name, truth, pix, matched_hat, random_hat in zip(
        PARAM_NAMES,
        expected,
        np.asarray(cond["params"], dtype=float),
        matched["estimated_params"],
        random["estimated_params"],
    ):
        row[f"pixel_{name}"] = float(pix)
        row[f"true_latent_{name}"] = float(truth)
        row[f"matched_est_{name}"] = float(matched_hat)
        row[f"random_est_{name}"] = float(random_hat)
    rows.append(row)

    if index == 1 or index == len(jobs) or index % max(1, len(jobs) // 5) == 0:
        elapsed = time.perf_counter() - reg_started
        eta = elapsed / index * (len(jobs) - index)
        print(
            f"[{index}/{len(jobs)}] sample {sample_id} {cond['name']} | "
            f"matched bit={row['matched_bit_accuracy']:.3f} random={row['random_bit_accuracy']:.3f} | "
            f"ang_err={row['matched_angle_err']:.2f} | "
            f"elapsed {fmt_duration(elapsed)} ~{fmt_duration(eta)} left",
            flush=True,
        )

results = pd.DataFrame(rows)
results.to_csv(OUTPUT_DIR / "trial_results.csv", index=False)
print(f"wrote {OUTPUT_DIR / 'trial_results.csv'} ({len(results)} rows)")
print(f"registration wall {fmt_duration(time.perf_counter() - reg_started)}")


## 6. Summaries and plots


In [ ]:
summary = (
    results.groupby("condition", observed=True)
    .agg(
        n=("sample_id", "count"),
        matched_bit_accuracy=("matched_bit_accuracy", "mean"),
        random_bit_accuracy=("random_bit_accuracy", "mean"),
        random_bit_accuracy_at_true=("random_bit_accuracy_at_true", "mean"),
        matched_bit_correlation=("matched_bit_correlation", "mean"),
        matched_continuous_correlation=("matched_continuous_correlation", "mean"),
        matched_param_mae=("matched_param_mae", "mean"),
        random_param_mae=("random_param_mae", "mean"),
        matched_angle_err=("matched_angle_err", "mean"),
        matched_tx_err=("matched_tx_err", "mean"),
        matched_ty_err=("matched_ty_err", "mean"),
        random_angle_err=("random_angle_err", "mean"),
        mean_true_angle=("true_latent_angle_deg", "mean"),
        mean_matched_angle=("matched_est_angle_deg", "mean"),
        mean_true_tx=("true_latent_tx", "mean"),
        mean_matched_tx=("matched_est_tx", "mean"),
        mean_true_ty=("true_latent_ty", "mean"),
        mean_matched_ty=("matched_est_ty", "mean"),
        mean_true_param_distance=("true_param_distance", "mean"),
        corr_zhat_vs_z=("corr_zhat_vs_z", "mean"),
    )
    .reset_index()
)
order = {c["name"]: i for i, c in enumerate(CONDITIONS)}
summary["ord"] = summary.condition.map(order)
summary = summary.sort_values("ord").drop(columns=["ord"]).reset_index(drop=True)
summary.to_csv(OUTPUT_DIR / "summary_by_condition.csv", index=False)
print(summary.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
x = np.arange(len(summary))
axes[0].bar(x - 0.17, summary.matched_bit_accuracy, width=0.34, label="matched")
axes[0].bar(x + 0.17, summary.random_bit_accuracy, width=0.34, label="random")
axes[0].axhline(0.5, color="0.4", ls=":", label="chance")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary.condition, rotation=25, ha="right")
axes[0].set_ylim(0.45, 1.02)
axes[0].set_ylabel("bit accuracy")
axes[0].set_title("Bit accuracy after inverse-warp")
axes[0].legend(fontsize=8)

axes[1].bar(x - 0.17, summary.matched_param_mae, width=0.34, label="matched")
axes[1].bar(x + 0.17, summary.random_param_mae, width=0.34, label="random")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary.condition, rotation=25, ha="right")
axes[1].set_ylabel("mean |param| error")
axes[1].set_title("Latent parameter MAE vs ideal map")
axes[1].legend(fontsize=8)
fig.suptitle(f"DDIM random-affine sign-template ({NUM_SAMPLES} samples)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "bits_and_param_error_by_condition.png", bbox_inches="tight")
plt.show()

nonbase = results.loc[results.condition != "baseline"].copy()
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
axes[0].scatter(nonbase.true_latent_angle_deg, nonbase.matched_est_angle_deg, s=40, label="matched")
axes[0].scatter(nonbase.true_latent_angle_deg, nonbase.random_est_angle_deg, s=28, marker="x", label="random")
lim_a = (ANGLE_RANGE[0] - 1, ANGLE_RANGE[1] + 1)
axes[0].plot(lim_a, lim_a, ls="--", color="0.3")
axes[0].set_xlim(lim_a)
axes[0].set_ylim(lim_a)
axes[0].set_xlabel("expected latent angle")
axes[0].set_ylabel("estimate")
axes[0].set_title("Angle")
axes[0].legend(fontsize=8)

lim_t = (TRANSLATION_RANGE[0] - 0.25, TRANSLATION_RANGE[1] + 0.25)
axes[1].scatter(nonbase.true_latent_tx, nonbase.matched_est_tx, s=40, label="matched")
axes[1].scatter(nonbase.true_latent_tx, nonbase.random_est_tx, s=28, marker="x", label="random")
axes[1].plot(lim_t, lim_t, ls="--", color="0.3")
axes[1].set_xlim(lim_t)
axes[1].set_ylim(lim_t)
axes[1].set_xlabel("expected latent tx")
axes[1].set_title("tx")
axes[1].legend(fontsize=8)

axes[2].scatter(nonbase.true_latent_ty, nonbase.matched_est_ty, s=40, label="matched")
axes[2].scatter(nonbase.true_latent_ty, nonbase.random_est_ty, s=28, marker="x", label="random")
axes[2].plot(lim_t, lim_t, ls="--", color="0.3")
axes[2].set_xlim(lim_t)
axes[2].set_ylim(lim_t)
axes[2].set_xlabel("expected latent ty")
axes[2].set_title("ty")
axes[2].legend(fontsize=8)
fig.suptitle("Estimated vs expected latent affine parameters (random draws)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "estimated_vs_expected_params.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.scatter(nonbase.true_param_distance, nonbase.matched_param_mae, s=45, label="matched")
ax.scatter(nonbase.true_param_distance, nonbase.random_param_mae, s=30, marker="x", label="random")
ax.set_xlabel("normalized distance of true params from identity")
ax.set_ylabel("parameter MAE")
ax.set_title("Recovery error vs warp magnitude")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "param_error_vs_magnitude.png", bbox_inches="tight")
plt.show()


## 7. Interpretation


In [ ]:
from experiments.generate_and_invert import _git_commit

meta_path = OUTPUT_DIR / "metadata.json"
meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.is_file() else {}
meta.update(
    {
        "registration": "sign-only bits of baseline z_hat vs full condition z_hat; random 0/1 control",
        "expected_latent_map": "same linear params; translation_px / vae_scale_factor",
        "perturbations": "uniform random affines inside latent search bounds; shared across samples",
        "n_samples_registered": NUM_SAMPLES,
        "trial_rows": int(len(results)),
        "git_commit": meta.get("git_commit") or _git_commit(),
    }
)
meta_path.write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")

print("Interpretation")
base = results.loc[results.condition == "baseline"]
print(
    f"- Baseline matched bit accuracy={base.matched_bit_accuracy.mean():.3f}; "
    f"random={base.random_bit_accuracy.mean():.3f}."
)
print(
    f"- Random-affine matched bit accuracy={nonbase.matched_bit_accuracy.mean():.3f}; "
    f"random-control={nonbase.random_bit_accuracy.mean():.3f}; "
    f"random-control at true={nonbase.random_bit_accuracy_at_true.mean():.3f}."
)
print(
    f"- Random-affine matched bit correlation={nonbase.matched_bit_correlation.mean():.3f}; "
    f"continuous corr(B, inverse-warp Y)={nonbase.matched_continuous_correlation.mean():.3f}."
)
print(
    f"- Random-affine matched param MAE={nonbase.matched_param_mae.mean():.3f}; "
    f"random-control MAE={nonbase.random_param_mae.mean():.3f}."
)
for name in [c["name"] for c in CONDITIONS if c["name"] != "baseline"]:
    block = results.loc[results.condition == name]
    pix = next(c["params"] for c in CONDITIONS if c["name"] == name)
    print(
        f"- {name} pixel={tuple(round(float(v), 3) for v in pix)}: "
        f"matched ang/tx/ty err="
        f"{block.matched_angle_err.mean():.3f}/{block.matched_tx_err.mean():.3f}/{block.matched_ty_err.mean():.3f}; "
        f"bit_acc={block.matched_bit_accuracy.mean():.3f} (control {block.random_bit_accuracy.mean():.3f})."
    )
gap = nonbase.matched_bit_accuracy.mean() - nonbase.random_bit_accuracy.mean()
print(f"- Matched-minus-control bit-accuracy gap: {gap:.3f}.")
print(f"- Artifacts under {OUTPUT_DIR}")
print(f"- Total wall since generate start ~{fmt_duration(time.perf_counter() - run_started)}")
